# 仓库粒度的冲突消解-数据集预处理

## 一.50Repo 数据集处理

### 1.top5 数量的仓库冲突内容提取

In [ ]:
import json
from collections import defaultdict

def count_by_repo_from_file(json_file):
    repo_counts = defaultdict(int)
    
    with open(json_file, 'r', encoding='utf-8') as file:
        data = json.load(file)

    for item in data:
        if item.get('res_region') == "null":
            continue  # 跳过res=null的条目
        
        total_lines = (item.get("a_contents", "").count('\n') + 1) + (item.get("o_contents", "").count('\n') + 1) + (item.get("b_contents", "").count('\n') + 1)
        if total_lines > 36: # 计算总行数
            continue

        if 'file_name' in item:
            repo = item['file_name']
        else:
            repo = 'no_repo'
        repo_counts[repo] += 1

    sorted_repo_counts = dict(sorted(repo_counts.items(), key=lambda item: item[1], reverse=True))
    return sorted_repo_counts


json_file = 'YOUR_DATA_DIR/mergeMinePython/json/50Repo/50RepoV1Pretty.json'
repo_stats = count_by_repo_from_file(json_file)

print("Repo 统计（按数量从多到少排序）：")
total_count = 0
for repo, count in repo_stats.items():
    print(f"{repo}: {count}")
    total_count += count

print(f"\n总共有 {total_count} 个条目")

In [ ]:
import json
import os
from utils import rewrite_DiffMarks
from collections import defaultdict

# 需要筛选的 repo 列表
target_repos = [
    # "./index_conflict_files/XCoLab/", #: 9221
    # "./index_conflict_files/accumulo/", #: 6570
    # "./index_conflict_files/OpenFaces/", #: 3380
    # "./index_conflict_files/elasticsearch/", #: 3060
    # "./index_conflict_files/lucene-solr/", #: 1797
    # "./index_conflict_files/jgralab/", #: 1703
    # "./index_conflict_files/ThingML/", #: 1521
    # "./index_conflict_files/wro4j/", # 1494
    # "./index_conflict_files/fred/" # 1261
    "./index_conflict_files/zk/", # : 1167
    "./index_conflict_files/realm-java/", # : 954
    "./index_conflict_files/UniversalMediaServer/" # : 901
]

def extract_and_save_data(json_file):
    repo_data = defaultdict(list)
    with open(json_file, 'r', encoding='utf-8') as file:
        data = json.load(file)
    
    for item in data:
        # 检查 res_label 是否为 "null"
        if item.get('res_region') == "null":
            continue

        # 计算总行数
        total_lines = (item.get("a_contents", "").count('\n') + 1) + (item.get("o_contents", "").count('\n') + 1) + (item.get("b_contents", "").count('\n') + 1)
        if total_lines > 36:
            continue

        if 'file_name' in item and item['file_name'] in target_repos:

            extracted_item = {
                "repo": item.get("file_name", ""),
                "res_label": item.get("res_label", ""),
                "id": item.get("id", ""),
                "json_name": item.get("json_name", ""),
                "token_level_result": rewrite_DiffMarks(item.get("token_level_result", "")),
                "a_contents": item.get("a_contents", ""),
                "o_contents": item.get("o_contents", ""),
                "b_contents": item.get("b_contents", ""),
                "res_region": item.get("res_region", "")
            }
            repo_data[item['file_name']].append(extracted_item)

    for repo_url, items in repo_data.items():
        repo_name = repo_url.rstrip('/').split('/')[-1]
        output_file = f"dataset/50repo/json/{repo_name}.json" # 写入 JSON 文件路径

        with open(output_file, 'w', encoding='utf-8') as outfile:
            json.dump(items, outfile, indent=4, ensure_ascii=False)
        
        print(f"已保存 {len(items)} 条数据到文件: {output_file}")

json_file = 'YOUR_DATA_DIR/mergeMinePython/json/50Repo/50RepoV1Pretty.json'
extract_and_save_data(json_file)

### 2.json data -> vector database

In [ ]:
import json

def json2vector(input_path, output_path):
    with open(input_path, 'r', encoding='utf-8') as file:
        data_list = json.load(file)

    valid_records_count = 0

    with open(output_path, 'w', encoding='utf-8') as output_file:
        for data in data_list:
            res_label = data.get('res_label', '')
            if res_label == 'null':
                continue
            a_contents = data.get('a_contents', '')
            o_contents = data.get('o_contents', '')
            b_contents = data.get('b_contents', '')
            res_region = data.get('res_region', '')
            global_id = data.get('id', '')
            
            conflict = 'concurrent_change\n' + a_contents + '\nbase_content\n' + o_contents + '\nincoming_change\n' + b_contents
            processed_conflict = conflict.replace('\n', ' _newline_ ').replace('\t', ' ')
            processed_res = res_region.replace('\n', ' _newline_ ').replace('\t', ' ')
            output_file.write(processed_conflict + '\t' + processed_res + '\t' + str(global_id) + '\n')
            valid_records_count += 1

    print(f"处理完毕，结果已写入到文件中。共处理了 {valid_records_count} 条有效记录。")

In [ ]:
json2vector(input_path = "dataset/50repo/json/accumulo.json", output_path = 'dataset/50repo/vector/accumulo_Conflicts.txt')
json2vector(input_path = "dataset/50repo/json/elasticsearch.json", output_path = 'dataset/50repo/vector/elasticsearch_Conflicts.txt')
json2vector(input_path = "dataset/50repo/json/jgralab.json", output_path = 'dataset/50repo/vector/jgralab_Conflicts.txt')
json2vector(input_path = "dataset/50repo/json/lucene-solr.json", output_path = 'dataset/50repo/vector/lucene_Conflicts.txt')
json2vector(input_path = "dataset/50repo/json/OpenFaces.json", output_path = 'dataset/50repo/vector/OpenFaces_Conflicts.txt')
json2vector(input_path = "dataset/50repo/json/ThingML.json", output_path = 'dataset/50repo/vector/ThingML_Conflicts.txt')
json2vector(input_path = "dataset/50repo/json/XCoLab.json", output_path = 'dataset/50repo/vector/XCoLab_Conflicts.txt')

In [ ]:
json2vector(input_path = "dataset/50repo/json/wro4j.json", output_path = 'dataset/50repo/vector/wro4j_Conflicts.txt')
json2vector(input_path = "dataset/50repo/json/fred.json", output_path = 'dataset/50repo/vector/fred_Conflicts.txt')

In [ ]:
json2vector(input_path = "dataset/50repo/json/zk.json", output_path = 'dataset/50repo/vector/zk_Conflicts.txt')
json2vector(input_path = "dataset/50repo/json/realm-java.json", output_path = 'dataset/50repo/vector/realm_Conflicts.txt')
json2vector(input_path = "dataset/50repo/json/UniversalMediaServer.json", output_path = 'dataset/50repo/vector/UniversalMediaServer_Conflicts.txt')

### 3.todo：方法签名 -> 信号量

In [ ]:
import ast
import hashlib
import os

# Define a normalized node type (using strings for simplicity)
class NormalizedNode:
    def __init__(self, kind, value=None):
        self.kind = kind
        self.value = value
    def __repr__(self):
      if self.value:
          return f"({self.kind}, {self.value})"
      else:
          return f"({self.kind})"

# Simple AST Normalizer class
class AstNormalizer(ast.NodeVisitor):
    def __init__(self):
        self.normalized_ast = []
        self.variable_mapping = {}
        self.variable_counter = 0

    def process_ident(self, id_node):
        # Change here: Get the name from the appropriate attribute
        name = id_node.id if isinstance(id_node, ast.Name) else id_node.arg
        if name in self.variable_mapping:
            canonical_name = self.variable_mapping[name]
            self.normalized_ast.append(NormalizedNode("ident", canonical_name))
        else:
            canonical_name = f"var{self.variable_counter}"
            self.variable_mapping[name] = canonical_name
            self.normalized_ast.append(NormalizedNode("ident", canonical_name))
            self.variable_counter += 1


    def visit_FunctionDef(self, node):
      self.normalized_ast.append(NormalizedNode("keyword", "function"))
      self.generic_visit(node)

    def visit_Name(self, node):
      self.process_ident(node)

    def visit_arg(self, node):
      self.process_ident(node)

    def visit_stmt(self, node):
       self.normalized_ast.append(NormalizedNode("statement", ast.unparse(node)))
       self.generic_visit(node)

    def visit_expr(self, node):
       self.normalized_ast.append(NormalizedNode("expression", ast.unparse(node)))
       self.generic_visit(node)

    def visit_block(self, node):
      self.normalized_ast.append(NormalizedNode("block", ast.unparse(node)))
      self.generic_visit(node)


# Function to normalize an AST
def normalize_ast(code):
    tree = ast.parse(code)
    normalizer = AstNormalizer()
    normalizer.visit(tree)
    return normalizer.normalized_ast


# Simple hash function for a normalized AST
def hash_normalized_ast(normalized_ast):
    hashable_string = str(normalized_ast) # Convert the list of normalized nodes to a string for hashing
    return hashlib.sha256(hashable_string.encode()).hexdigest()


def detect_homologous_functions(code):
    tree = ast.parse(code)
    function_hashes = {}

    for item in tree.body:
        if isinstance(item, ast.FunctionDef):
            normalized_ast = normalize_ast(ast.unparse(item))
            hash_value = hash_normalized_ast(normalized_ast)

            if hash_value not in function_hashes:
                function_hashes[hash_value] = []
            function_hashes[hash_value].append(item.name)

    print("Potential homologous functions:")
    for hash_val, functions in function_hashes.items():
        if len(functions) > 1:
            print(f"   Hash: {hash_val} Functions: {functions}")


def process_directory(directory_path):
    if os.path.isdir(directory_path):
        for entry in os.listdir(directory_path):
            file_path = os.path.join(directory_path, entry)
            if os.path.isfile(file_path):
                print(f"Analyzing file: {file_path}")
                with open(file_path, "r") as f:
                  code = f.read()
                  detect_homologous_functions(code)

    elif os.path.isfile(directory_path):
            print(f"Analyzing file: {directory_path}")
            with open(directory_path, "r") as f:
                code = f.read()
                detect_homologous_functions(code)
    else:
        print(f"Invalid file or directory: {directory_path}")

# Example usage
if __name__ == "__main__":
    # Create an example.py file in the current directory for testing
    example_code = """
def add(a, b):
    sum = a + b
    return sum

def sub(c, d):
    diff = c - d
    return diff

def sum_all(a, b):
    result = a + b
    return result

def sum_all_again(x, y):
    z = x + y
    return z
"""
    with open("example.py", "w") as f:
        f.write(example_code)
    process_directory("example.py")
    # To test on a directory containing several .py files, replace the path by the directory path in the process_directory method

## 二.Mergebert 数据集处理

### 1.mergebert数据集 repo分类

In [ ]:
import os
import json
from collections import defaultdict

# 先检查文件是否存在
test_paths = [
    'YOUR_DATA_DIR/mergeMinePython/json/mergeBert/javaContextPrettyVersionAll2.json',
    'YOUR_DATA_DIR/dataset/javaContextPrettyVersionAll2.json',
    './dataset/mergebert/json/javaContextPrettyVersionAll2.json'
]

print("检查文件路径：")
for path in test_paths:
    exists = os.path.exists(path)
    print(f"{path}: {'存在' if exists else '不存在'}")

def count_by_repo_from_file(json_file):
    repo_counts = defaultdict(int)
    
    with open(json_file, 'r', encoding='utf-8') as file:
        data = json.load(file)

    for item in data:
        if item.get('res_region') == "null":
            continue  # 跳过res=null的条目
        
        total_lines = (item.get("a_contents", "").count('\n') + 1) + (item.get("o_contents", "").count('\n') + 1) + (item.get("b_contents", "").count('\n') + 1)
        if total_lines > 36: # 计算总行数
            continue

        if 'repo' in item:
            repo = item['repo']
        else:
            repo = 'no_repo'
        repo_counts[repo] += 1

    sorted_repo_counts = dict(sorted(repo_counts.items(), key=lambda item: item[1], reverse=True))
    return sorted_repo_counts

# 请选择正确的文件路径
json_file = 'YOUR_DATA_DIR/mergeMinePython/json/mergeBert/javaContextPrettyVersionAll2.json'

if os.path.exists(json_file):
    repo_stats = count_by_repo_from_file(json_file)
    print("\nRepo 统计（按数量从多到少排序）：")
    total_count = 0
    for repo, count in repo_stats.items():
        print(f"{repo}: {count}")
        total_count += count
    print(f"\n总共有 {total_count} 个条目")
else:
    print(f"\n文件不存在：{json_file}")

FileNotFoundError: [Errno 2] No such file or directory: 'YOUR_DATA_DIR/mergeMinePython/json/mergeBert/javaContextPrettyVersionAll2.json'

In [ ]:
import json
import os
from utils import rewrite_DiffMarks
from collections import defaultdict

# 需要筛选的 repo 列表
target_repos = [
    # "https://github.com/RumbleDB/rumble", # : 1415
    # "https://github.com/Azure/azure-sdk-for-java", # : 958
    # "https://github.com/52North/SOS", # : 902
    # "https://github.com/Azure/autorest", # : 866
    # "https://github.com/molgenis/molgenis", # : 797
    # "https://github.com/sleuthkit/autopsy", # : 775
    # "https://github.com/micdoodle8/Galacticraft" # : 737
    # "https://github.com/apache/sis", # : 598
    # "https://github.com/spring-projects/spring-boot", # : 586
    # "https://github.com/apache/cloudstack" # : 533
    "https://github.com/orientechnologies/orientdb" #: 529
]

def extract_and_save_data(json_file):
    repo_data = defaultdict(list)
    with open(json_file, 'r', encoding='utf-8') as file:
        data = json.load(file)
    
    for item in data:
        # 检查 res_label 是否为 "null"
        if item.get('res_region') == "null":
            continue

        # 计算总行数
        total_lines = (item.get("a_contents", "").count('\n') + 1) + (item.get("o_contents", "").count('\n') + 1) + (item.get("b_contents", "").count('\n') + 1)
        if total_lines > 36:
            continue

        if 'repo' in item and item['repo'] in target_repos:

            # 根据 json_name 打开对应的 JSON 文件获取 commitHash
            json_path = 'YOUR_DATA_DIR/mergebert_dataset/fse2022/automated-analysis-data/Java/' + item.get("json_name", "")
            commit_hash = ""
            if json_path and os.path.exists(json_path):
                try:
                    with open(json_path, 'r', encoding='utf-8') as json_file_content:
                        json_content = json.load(json_file_content)
                        commit_hash = json_content.get("commitHash", "")
                except Exception as e:
                    print(f"无法读取文件 {json_path}: {e}")
                    commit_hash = ""
            else:
                commit_hash = ""

            extracted_item = {
                "file_name": item.get("file_name", ""),
                "repo": item.get("repo", ""),
                "commitHash": commit_hash,
                "res_label": item.get("res_label", ""),
                "id": item.get("id", ""),
                "json_name": item.get("json_name", ""),
                "token_level_result": rewrite_DiffMarks(item.get("token_level_result", "")),
                "a_contents": item.get("a_contents", ""),
                "o_contents": item.get("o_contents", ""),
                "b_contents": item.get("b_contents", ""),
                "res_region": item.get("res_region", "")
            }
            repo_data[item['repo']].append(extracted_item)

    for repo_url, items in repo_data.items():
        repo_name = repo_url.split('/')[-1]
        output_file = f"dataset/mergebert/json/{repo_name}.json" # 写入 JSON 文件路径

        with open(output_file, 'w', encoding='utf-8') as outfile:
            json.dump(items, outfile, indent=4, ensure_ascii=False)
        
        print(f"已保存 {len(items)} 条数据到文件: {output_file}")

json_file = 'YOUR_DATA_DIR/mergeMinePython/json/mergeBert/javaContextPrettyVersionAll2.json'
extract_and_save_data(json_file)

### 2.提取repo下commit message

In [ ]:
import json
from utils import execute

def get_message(json_file_path):
    try:
        # 读取JSON文件
        with open(json_file_path, 'r', encoding='utf-8') as file:
            json_data = json.load(file)
        
        # 只处理前10条数据
        for i, entry in enumerate(json_data):
            repo_url = entry.get("repo", "")
            commit_hash = entry.get("commitHash", "")
            
            if repo_url:
                path = "YOUR_DATA_DIR/gitRepo/" + repo_url.split("/")[-1]  # 获取 URL 最后一部分
            
            # 调用 execute 方法
            if path and commit_hash:
                # a, b, o
                a_message, b_message, o_message = execute(path, commit_hash)
                entry["a_message"] = a_message
                entry["b_message"] = b_message
                entry["o_message"] = o_message
            print(f'{i + 1} {a_message, b_message, o_message}')
            
        # 将更新后的数据写回文件
        with open(json_file_path, 'w', encoding='utf-8') as file:
            json.dump(json_data, file, indent=4, ensure_ascii=False)
        
        print(f"Processed the first 10 entries and saved to {json_file_path}")
    
    except FileNotFoundError:
        print(f"The file {json_file_path} does not exist.")
    except json.JSONDecodeError:
        print(f"Failed to decode JSON from the file {json_file_path}.")

In [ ]:
import json
from utils import execute_with_info as execute  # 返回 dict 或 None

def get_message_info(json_file_path):
    try:
        # 读取 JSON 文件
        with open(json_file_path, 'r', encoding='utf-8') as file:
            json_data = json.load(file)
        
        # 只处理前 10 条有效数据
        processed_count = 0
        for entry in json_data:
            # if processed_count >= 10:
            #     break

            repo_url = entry.get("repo", "")
            commit_hash = entry.get("commitHash", "")
            
            if not repo_url or not commit_hash:
                continue  # 跳过无效条目

            path = "YOUR_DATA_DIR/gitRepo/" + repo_url.split("/")[-1]

            if not path or not commit_hash:
                continue

            # 调用 execute —— 现在它返回结构化 info 或 None
            result = execute(path, commit_hash)

            if result is not None:
                entry["a_info"] = result["parent1"]      # 包含 message + time
                entry["b_info"] = result["parent2"]
                entry["o_info"] = result["common_ancestor"]
                processed_count += 1
                print(f'{processed_count}: Successfully processed {commit_hash}')
                print(f'{result["parent1"], result["parent2"], result["common_ancestor"]}')
            else:
                # 可选：记录失败，或留空
                entry["a_info"] = None
                entry["b_info"] = None
                entry["o_info"] = None
                print(f'{processed_count + 1}: Failed to process {commit_hash}')

        # 将更新后的数据写回文件
        with open(json_file_path, 'w', encoding='utf-8') as file:
            json.dump(json_data, file, indent=4, ensure_ascii=False)
        
        print(f"Processed {processed_count} entries and saved to {json_file_path}")
    
    except FileNotFoundError:
        print(f"The file {json_file_path} does not exist.")
    except json.JSONDecodeError:
        print(f"Failed to decode JSON from the file {json_file_path}.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

In [ ]:
get_message_info("YOUR_PROJECT_DIR/dataset/mergebert/json/autopsy_copy.json")

In [ ]:
get_message("YOUR_DATA_DIR/repoMerge/dataset/mergebert/json/autorest.json")

In [ ]:
get_message("YOUR_DATA_DIR/repoMerge/dataset/mergebert/json/azure-sdk-for-java.json")

In [ ]:
get_message("YOUR_DATA_DIR/repoMerge/dataset/mergebert/json/Galacticraft.json")

In [ ]:
get_message("YOUR_DATA_DIR/repoMerge/dataset/mergebert/json/molgenis.json")

In [ ]:
get_message("YOUR_DATA_DIR/repoMerge/dataset/mergebert/json/rumble.json")

In [ ]:
get_message("YOUR_DATA_DIR/repoMerge/dataset/mergebert/json/SOS.json")

In [ ]:
get_message("YOUR_DATA_DIR/repoMerge/dataset/mergebert/json/sis.json")

In [ ]:
get_message("YOUR_DATA_DIR/repoMerge/dataset/mergebert/json/spring-boot.json")

In [ ]:
get_message("YOUR_DATA_DIR/repoMerge/dataset/mergebert/json/cloudstack.json")

### 3.json data -> vector database

In [1]:
import json

def json_to_vector(input_path, output_path):
    with open(input_path, 'r', encoding='utf-8') as file:
        data_list = json.load(file)

    valid_records_count = 0

    with open(output_path, 'w', encoding='utf-8') as output_file:
        for data in data_list:
            res_label = data.get('res_label', '')
            if res_label == 'null':
                continue
            a_contents = data.get('a_contents', '')
            o_contents = data.get('o_contents', '')
            b_contents = data.get('b_contents', '')
            res_region = data.get('res_region', '')
            global_id = data.get('id', '')
            
            conflict = 'concurrent_change\n' + a_contents + '\nbase_content\n' + o_contents + '\nincoming_change\n' + b_contents
            processed_conflict = conflict.replace('\n', ' _newline_ ').replace('\t', ' ')
            processed_res = res_region.replace('\n', ' _newline_ ').replace('\t', ' ')
            output_file.write(processed_conflict + '\t' + processed_res + '\t' + str(global_id) + '\n')
            valid_records_count += 1

    print(f"处理完毕，结果已写入到文件中。共处理了 {valid_records_count} 条有效记录。")

In [2]:
json_to_vector(input_path = "dataset\mergebert_80\mergebert_80.json", output_path = 'dataset/mergebert/vector/mergebert_80_Conflicts.txt')

处理完毕，结果已写入到文件中。共处理了 6674 条有效记录。


In [ ]:
json_to_vector(input_path = "dataset/mergebert/json/autopsy_copy.json", output_path = 'dataset/mergebert/vector/autopsy_copy_Conflicts.txt')

In [ ]:
json_to_vector(input_path = "dataset/mergebert/json/autopsy.json", output_path = 'dataset/mergebert/vector/autopsy_Conflicts.txt')
json_to_vector(input_path = "dataset/mergebert/json/autorest.json", output_path = 'dataset/mergebert/vector/autorest_Conflicts.txt')
json_to_vector(input_path = "dataset/mergebert/json/azure-sdk-for-java.json", output_path = 'dataset/mergebert/vector/azure_Conflicts.txt')
json_to_vector(input_path = "dataset/mergebert/json/Galacticraft.json", output_path = 'dataset/mergebert/vector/Galacticraft_Conflicts.txt')
json_to_vector(input_path = "dataset/mergebert/json/molgenis.json", output_path = 'dataset/mergebert/vector/molgenis_Conflicts.txt')
json_to_vector(input_path = "dataset/mergebert/json/rumble.json", output_path = 'dataset/mergebert/vector/rumble_Conflicts.txt')
json_to_vector(input_path = "dataset/mergebert/json/SOS.json", output_path = 'dataset/mergebert/vector/SOS_Conflicts.txt')

In [ ]:
json_to_vector(input_path = "dataset/mergebert/json/sis.json", output_path = 'dataset/mergebert/vector/sis_Conflicts.txt')
json_to_vector(input_path = "dataset/mergebert/json/spring-boot.json", output_path = 'dataset/mergebert/vector/spring_Conflicts.txt')
json_to_vector(input_path = "dataset/mergebert/json/cloudstack.json", output_path = 'dataset/mergebert/vector/cloudstack_Conflicts.txt')

In [ ]:
json_to_vector(input_path = "dataset/mergebert/json/orientdb.json", output_path = 'dataset/mergebert/vector/orientdb_Conflicts.txt')

## 三.Mergebert 非Java数据集处理

In [ ]:
import json
from collections import defaultdict

def count_by_repo_from_file(json_file):
    """
    统计 JSON 文件中每个 repo 的条目数量，并跳过不符合条件的条目。
    
    参数:
        json_file (str): JSON 文件的路径。
        
    返回:
        dict: 按照 repo 条目数量从多到少排序的统计结果。
    """
    repo_counts = defaultdict(int)
    
    # 读取 JSON 文件
    with open(json_file, 'r', encoding='utf-8') as file:
        data = json.load(file)

    # 遍历数据并统计符合条件的条目
    for item in data:
        if item.get('res_region') == "null":
            continue  # 跳过 res_region=null 的条目
        
        # 计算总行数
        total_lines = (
            (item.get("a_contents", "").count('\n') + 1) +
            (item.get("o_contents", "").count('\n') + 1) +
            (item.get("b_contents", "").count('\n') + 1)
        )
        if total_lines > 36:  # 如果总行数超过 36，则跳过
            continue

        # 获取 repo 名称，如果不存在则设置为 'no_repo'
        repo = item.get('repo', 'no_repo')
        repo_counts[repo] += 1

    # 按条目数量从多到少排序
    sorted_repo_counts = dict(sorted(repo_counts.items(), key=lambda item: item[1], reverse=True))
    return sorted_repo_counts


def print_repo_stats(json_file):
    """
    打印 JSON 文件中每个 repo 的统计信息。
    
    参数:
        json_file (str): JSON 文件的路径。
    """
    # 调用统计方法
    repo_stats = count_by_repo_from_file(json_file)

    print("Repo 统计（按数量从多到少排序）：")
    total_count = 0
    for repo, count in repo_stats.items():
        print(f"{repo}: {count}")
        total_count += count

    print(f"\n总共有 {total_count} 个条目")

In [ ]:
"""提取到json"""
import json
import os
from utils import rewrite_DiffMarks
from collections import defaultdict

def extract_and_save_data(json_file, target_repos):
    repo_data = defaultdict(list)
    with open(json_file, 'r', encoding='utf-8') as file:
        data = json.load(file)
    
    for item in data:
        # 检查 res_label 是否为 "null"
        if item.get('res_region') == "null" or item.get('res_region') is None:
            continue

        # 计算总行数
        total_lines = (item.get("a_contents", "").count('\n') + 1) + (item.get("o_contents", "").count('\n') + 1) + (item.get("b_contents", "").count('\n') + 1)
        if total_lines > 36:
            continue

        if 'repo' in item and item['repo'] in target_repos:

            extracted_item = {
                "file_name": item.get("file_name", ""),
                "repo": item.get("repo", ""),
                "commitHash": item.get("commitHash", ""),
                "res_label": item.get("res_label", ""),
                "id": item.get("id", ""),
                "json_name": item.get("json_name", ""),
                # "token_level_result": rewrite_DiffMarks(item.get("token_level_result", "")),
                "a_contents": item.get("a_contents", ""),
                "o_contents": item.get("o_contents", ""),
                "b_contents": item.get("b_contents", ""),
                "res_region": item.get("res_region", "")
            }
            repo_data[item['repo']].append(extracted_item)

    for repo_url, items in repo_data.items():
        repo_name = repo_url.split('/')[-1]
        output_file = f"dataset/mergebert/json_ts/{repo_name}.json" # 写入 JSON 文件路径

        with open(output_file, 'w', encoding='utf-8') as outfile:
            json.dump(items, outfile, indent=4, ensure_ascii=False)
        
        print(f"已保存 {len(items)} 条数据到文件: {output_file}")

In [ ]:
"""提取到json 24000个字符"""
import json
import os
from utils import rewrite_DiffMarks
from collections import defaultdict

def extract_and_save_data_24000(json_file, target_repos):
    repo_data = defaultdict(list)
    with open(json_file, 'r', encoding='utf-8') as file:
        data = json.load(file)
    
    for item in data:
        # 检查 res_label 是否为 "null"
        if item.get('res_region') == "null" or item.get('res_region') is None:
            continue

        # 计算总行数
        total_lines = (item.get("a_contents", "").count('\n') + 1) + \
                      (item.get("o_contents", "").count('\n') + 1) + \
                      (item.get("b_contents", "").count('\n') + 1)
        if total_lines > 36:
            continue

        # 计算总字符数
        total_chars = len(item.get("a_contents", "")) + \
                      len(item.get("o_contents", "")) + \
                      len(item.get("b_contents", ""))
        if total_chars > 24000:
            continue

        if 'repo' in item and item['repo'] in target_repos:

            extracted_item = {
                "file_name": item.get("file_name", ""),
                "repo": item.get("repo", ""),
                "commitHash": item.get("commitHash", ""),
                "res_label": item.get("res_label", ""),
                "id": item.get("id", ""),
                "json_name": item.get("json_name", ""),
                # "token_level_result": rewrite_DiffMarks(item.get("token_level_result", "")),
                "a_contents": item.get("a_contents", ""),
                "o_contents": item.get("o_contents", ""),
                "b_contents": item.get("b_contents", ""),
                "res_region": item.get("res_region", "")
            }
            repo_data[item['repo']].append(extracted_item)

    for repo_url, items in repo_data.items():
        repo_name = repo_url.split('/')[-1]
        output_file = f"dataset/mergebert/json_js/{repo_name}.json"  # 写入 JSON 文件路径

        with open(output_file, 'w', encoding='utf-8') as outfile:
            json.dump(items, outfile, indent=4, ensure_ascii=False)
        
        print(f"已保存 {len(items)} 条数据到文件: {output_file}")

In [ ]:
"""json data -> vector database"""
import json

def json_to_vector(input_path, output_path):
    with open(input_path, 'r', encoding='utf-8') as file:
        data_list = json.load(file)

    valid_records_count = 0

    with open(output_path, 'w', encoding='utf-8') as output_file:
        for data in data_list:
            res_label = data.get('res_label', '')
            if res_label == 'null':
                continue
            a_contents = data.get('a_contents', '')
            o_contents = data.get('o_contents', '')
            b_contents = data.get('b_contents', '')
            res_region = data.get('res_region', '')
            global_id = data.get('id', '')
            
            conflict = 'concurrent_change\n' + a_contents + '\nbase_content\n' + o_contents + '\nincoming_change\n' + b_contents
            processed_conflict = conflict.replace('\n', ' _newline_ ').replace('\t', ' ')
            processed_res = res_region.replace('\n', ' _newline_ ').replace('\t', ' ')
            output_file.write(processed_conflict + '\t' + processed_res + '\t' + str(global_id) + '\n')
            valid_records_count += 1

    print(f"处理完毕，结果已写入到文件中。共处理了 {valid_records_count} 条有效记录。")

#### 1. C#数据集

In [ ]:
import os
import json

# 输入目录路径
input_dir = "YOUR_DATA_DIR/mergebert_dataset/fse2022/automated-analysis-data/CSharp/"

# 输出文件路径
output_file_path = "YOUR_DATA_DIR/repoMerge/dataset/mergebert/json_cs/CSharpAllConflicts.json"

# 遍历目录，筛选出以 _metadata.json 结尾的文件
json_files = [f for f in os.listdir(input_dir) if f.endswith("_metadata.json")]

# 初始化输出列表和全局 ID 计数器
output = []
global_id = 1

# 遍历所有符合条件的 JSON 文件
for json_file_name in json_files:
    # 构造完整的文件路径
    json_file_path = os.path.join(input_dir, json_file_name)
    
    # 读取 JSON 数据
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # 提取 conflict_chunks 信息
    conflicting_chunks = data.get("conflicting_chunks", [])
    
    for chunk in conflicting_chunks:
        # 获取基本信息
        file_name = data.get("fname", "")
        repo = data.get("repo", "")
        commit_hash = data.get("commitHash", "")
        res_label = chunk.get("label", "")

        # 构造每个冲突块的字典
        conflict_block = {
            "file_name": file_name,
            "repo": repo,
            "commitHash": commit_hash,
            "res_label": res_label,
            "id": global_id,  # 使用全局递增 ID
            "json_name": json_file_path,  # 保存文件路径
            "a_contents": chunk.get("a_contents", ""),
            "o_contents": chunk.get("base_contents", ""),  # o_contents 对应 base_contents
            "b_contents": chunk.get("b_contents", ""),
            "res_region": chunk.get("res_region", "")
        }
        output.append(conflict_block)

        # 更新全局 ID
        global_id += 1

# 将结果保存到输出文件
with open(output_file_path, "w", encoding='utf-8') as outfile:
    json.dump(output, outfile, indent=4)

print(f"处理完成，共收集了 {len(output)} 个冲突块，保存至 {output_file_path}")

In [ ]:
"""数据repo粒度分析统计"""
json_file = 'YOUR_DATA_DIR/repoMerge/dataset/mergebert/json_cs/CSAllConflicts.json'
print_repo_stats(json_file)

In [ ]:
# 需要筛选的 repo 列表
target_repos = [
    "https://github.com/dotnet/msbuild", #: 3001
    "https://github.com/aws/aws-sdk-net", #: 1551
    "https://github.com/nopSolutions/nopCommerce", #: 1027
    "https://github.com/AElfProject/AElf" #: 689
]

json_file = 'YOUR_DATA_DIR/repoMerge/dataset/mergebert/json_cs/CSAllConflicts.json'
extract_and_save_data(json_file, target_repos)

In [ ]:
json_to_vector("dataset/mergebert/json_cs/msbuild.json", "dataset/mergebert/json_cs/vector/msbuild_Conflicts.txt")
json_to_vector("dataset/mergebert/json_cs/aws-sdk-net.json", "dataset/mergebert/json_cs/vector/aws_Conflicts.txt")
json_to_vector("dataset/mergebert/json_cs/nopCommerce.json", "dataset/mergebert/json_cs/vector/nopCommerce_Conflicts.txt")
json_to_vector("dataset/mergebert/json_cs/AElf.json", "dataset/mergebert/json_cs/vector/AElf_Conflicts.txt")

#### 2. javaScript

In [ ]:
import os
import json

# 输入目录路径
input_dir = "YOUR_DATA_DIR/mergebert_dataset/fse2022/automated-analysis-data/JavaScript/"

# 输出文件路径
output_file_path = "YOUR_DATA_DIR/repoMerge/dataset/mergebert/json_js/JSAllConflicts.json"

# 遍历目录，筛选出以 _metadata.json 结尾的文件
json_files = [f for f in os.listdir(input_dir) if f.endswith("_metadata.json")]

# 初始化输出列表和全局 ID 计数器
output = []
global_id = 1

# 遍历所有符合条件的 JSON 文件
for json_file_name in json_files:
    # 构造完整的文件路径
    json_file_path = os.path.join(input_dir, json_file_name)
    
    # 读取 JSON 数据
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # 提取 conflict_chunks 信息
    conflicting_chunks = data.get("conflicting_chunks", [])
    
    for chunk in conflicting_chunks:
        # 获取基本信息
        file_name = data.get("fname", "")
        repo = data.get("repo", "")
        commit_hash = data.get("commitHash", "")
        res_label = chunk.get("label", "")

        # 构造每个冲突块的字典
        conflict_block = {
            "file_name": file_name,
            "repo": repo,
            "commitHash": commit_hash,
            "res_label": res_label,
            "id": global_id,  # 使用全局递增 ID
            "json_name": json_file_path,  # 保存文件路径
            "a_contents": chunk.get("a_contents", ""),
            "o_contents": chunk.get("base_contents", ""),  # o_contents 对应 base_contents
            "b_contents": chunk.get("b_contents", ""),
            "res_region": chunk.get("res_region", "")
        }
        output.append(conflict_block)

        # 更新全局 ID
        global_id += 1

# 将结果保存到输出文件
with open(output_file_path, "w", encoding='utf-8') as outfile:
    json.dump(output, outfile, indent=4)

print(f"处理完成，共收集了 {len(output)} 个冲突块，保存至 {output_file_path}")

In [ ]:
"""数据repo粒度分析统计"""
json_file = 'YOUR_DATA_DIR/repoMerge/dataset/mergebert/json_js/JSAllConflicts.json'
print_repo_stats(json_file)

In [ ]:
# 需要筛选的 repo 列表
target_repos = [
    "https://github.com/Yoast/javascript", #: 1090
    "https://github.com/highcharts/highcharts", #: 1008
    "https://github.com/Yoast/YoastSEO.js", #: 964
    "https://github.com/zixaphir/appchan-x" # : 917
]

json_file = 'YOUR_DATA_DIR/repoMerge/dataset/mergebert/json_js/JSAllConflicts.json'
extract_and_save_data(json_file, target_repos)

In [ ]:
json_to_vector("dataset/mergebert/json_js/highcharts.json", "dataset/mergebert/json_js/vector/highcharts_Conflicts.txt")
json_to_vector("dataset/mergebert/json_js/appchan-x.json", "dataset/mergebert/json_js/vector/appchan_Conflicts.txt")
json_to_vector("dataset/mergebert/json_js/YoastSEO.js.json", "dataset/mergebert/json_js/vector/YoastSEO.js_Conflicts.txt")
json_to_vector("dataset/mergebert/json_js/javascript.json", "dataset/mergebert/json_js/vector/javascript_Conflicts.txt")

In [ ]:
# 需要筛选的 repo 列表
target_repos = [
    "https://github.com/zixaphir/appchan-x" # : 917
]

json_file = 'YOUR_DATA_DIR/repoMerge/dataset/mergebert/json_js/JSAllConflicts.json'
extract_and_save_data_24000(json_file, target_repos)

In [ ]:
json_to_vector("dataset/mergebert/json_js/appchan-x.json", "dataset/mergebert/json_js/vector/appchan_Conflicts.txt")

#### 3. TypeScript

In [ ]:
import os
import json

# 输入目录路径
input_dir = "YOUR_DATA_DIR/mergebert_dataset/fse2022/automated-analysis-data/TypeScript/"

# 输出文件路径
output_file_path = "YOUR_DATA_DIR/repoMerge/dataset/mergebert/json_ts/TSAllConflicts.json"

# 遍历目录，筛选出以 _metadata.json 结尾的文件
json_files = [f for f in os.listdir(input_dir) if f.endswith("_metadata.json")]

# 初始化输出列表和全局 ID 计数器
output = []
global_id = 1

# 遍历所有符合条件的 JSON 文件
for json_file_name in json_files:
    # 构造完整的文件路径
    json_file_path = os.path.join(input_dir, json_file_name)
    
    # 读取 JSON 数据
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # 提取 conflict_chunks 信息
    conflicting_chunks = data.get("conflicting_chunks", [])
    
    for chunk in conflicting_chunks:
        # 获取基本信息
        file_name = data.get("fname", "")
        repo = data.get("repo", "")
        commit_hash = data.get("commitHash", "")
        res_label = chunk.get("label", "")

        # 构造每个冲突块的字典
        conflict_block = {
            "file_name": file_name,
            "repo": repo,
            "commitHash": commit_hash,
            "res_label": res_label,
            "id": global_id,  # 使用全局递增 ID
            "json_name": json_file_path,  # 保存文件路径
            "a_contents": chunk.get("a_contents", ""),
            "o_contents": chunk.get("base_contents", ""),  # o_contents 对应 base_contents
            "b_contents": chunk.get("b_contents", ""),
            "res_region": chunk.get("res_region", "")
        }
        output.append(conflict_block)

        # 更新全局 ID
        global_id += 1

# 将结果保存到输出文件
with open(output_file_path, "w", encoding='utf-8') as outfile:
    json.dump(output, outfile, indent=4)

print(f"处理完成，共收集了 {len(output)} 个冲突块，保存至 {output_file_path}")

In [ ]:
"""数据repo粒度分析统计"""
json_file = 'YOUR_DATA_DIR/repoMerge/dataset/mergebert/json_ts/TSAllConflicts.json'
print_repo_stats(json_file)

In [ ]:
# 需要筛选的 repo 列表
target_repos = [
    "https://github.com/Azure/autorest.typescript", #: 745
    "https://github.com/MaartenBaert/ssr", #: 677
    "https://github.com/nrkno/tv-automation-server-core", #: 502
    "https://github.com/cloudfoundry/stratos" #: 428
]

json_file = 'YOUR_DATA_DIR/repoMerge/dataset/mergebert/json_ts/TSAllConflicts.json'
extract_and_save_data(json_file, target_repos)

In [ ]:
json_to_vector("dataset/mergebert/json_ts/stratos.json", "dataset/mergebert/json_ts/vector/stratos_Conflicts.txt")
json_to_vector("dataset/mergebert/json_ts/tv-automation-server-core.json", "dataset/mergebert/json_ts/vector/tv-automation-server-core_Conflicts.txt")
json_to_vector("dataset/mergebert/json_ts/ssr.json", "dataset/mergebert/json_ts/vector/ssr_Conflicts.txt")
json_to_vector("dataset/mergebert/json_ts/autorest.typescript.json", "dataset/mergebert/json_ts/vector/autorest.typescript_Conflicts.txt")